In [ ]:
import pandas as pd
import numpy as np
from statsmodels.stats.stattools import jarque_bera
from statsmodels.stats.diagnostic import acorr_ljungbox
import statsmodels.api as sm
from statsmodels.stats import weightstats as stests
from statsmodels.stats import diagnostic as diag
from statsmodels.stats.sandwich_covariance import cov_hac
from statsmodels.regression.linear_model import OLS
import matplotlib.pyplot as plt
from scipy import stats


In [ ]:
# 1a)
# Load data
factors = pd.read_excel('Assignment5Data.xlsx', sheet_name='FamaFrench5Factors')
portfolios = pd.read_excel('Assignment5Data.xlsx', sheet_name='FamaFrench5x5')


# Merge data on time
data = pd.merge(factors, portfolios, on=['Year', 'Month'])
data.columns = data.columns.astype(str) # Set columns as strings


# Calculate returns
factor_columns = ['Mkt-Rf', 'SMB', 'HML', 'RMW', 'CMA']
data[factor_columns] = data[factor_columns]
portfolio_columns = ['11','12', '13', '14', '15', '21', '22', '23', '24', '25', '31', '32', '33', '34', '35',
                     '41', '42', '43', '44', '45', '51', '52', '53', '54', '55']
data[portfolio_columns] = data[portfolio_columns]
#data['RF'] = data['RF'] / 100

# Excess returns for factors and portfolios
for col in portfolio_columns:
    data[col + '_excess'] = data[col] - data['RF']

for col in factor_columns:
    data[col + '_excess'] = data[col] #- data['RF']

# Mean and stdev for the 5x5 portfolios
portfolio_excess_columns = [col + '_excess' for col in portfolio_columns]
portfolio_means = data[portfolio_excess_columns].mean()
portfolio_std = data[portfolio_excess_columns].std()

# Mean and stdev for factor portfolios
factor_excess_columns = [col + '_excess' for col in factor_columns]
factor_means = data[factor_excess_columns].mean()
factor_std = data[factor_excess_columns].std()

# Newey-West function
# NW function
def newey_west_se(series, lag=1):
    series = series.dropna()
    T = len(series)
    mean_adj = series - np.mean(series)
    
    # Variance term
    variance = np.sum(mean_adj**2) / T
    
    # Covariance terms
    covariances = 0
    for l in range(1, lag + 1):
        weight = 1 - l / (lag + 1)  # Bartlett weights
        cov = np.sum(mean_adj[l:] * mean_adj[:-l]) / T
        covariances += 2 * weight * cov
    
    # Newey-West variance
    nw_variance = variance + covariances
    nw_se = np.sqrt(nw_variance / T)
    return nw_se

# NW SE for factors and portfolios
nw_se_portfolios = {}
for col in portfolio_excess_columns:
    returns = data[col].dropna()
    nw_se_portfolios[col] = newey_west_se(returns, lag=1)

nw_se_factors = {}
for col in factor_excess_columns:
    returns = data[col].dropna()
    nw_se_factors[col] = newey_west_se(returns, lag=1)

# Results
# For Portfolios
portfolio_stats = pd.DataFrame({
    'Mean': portfolio_means,
    'Std Dev': portfolio_std,
    'NW SE Mean': pd.Series(nw_se_portfolios).values.flatten()
}, index=portfolio_means.index)

# For Factors
factor_stats = pd.DataFrame({
    'Mean': factor_means,
    'Std Dev': factor_std,
    'NW SE Mean': pd.Series(nw_se_factors).values.flatten()
}, index=factor_means.index)

print("Portfolio Statistics (Excess Returns):")
print(portfolio_stats)

print("\nFactor Statistics:")
print(factor_stats)

In [ ]:
data.tail()

In [ ]:
# 1b)

# Setup data storage
alphas = []
betas = []
tstats_alpha = []
tstats_beta = []
residuals = []

# Run regressions
for col in portfolio_excess_columns:
    y = data[col]
    x = data['Mkt-Rf']
    x = sm.add_constant(x)
    model = sm.OLS(y, x).fit(cov_type='HAC', cov_kwds={'maxlags':1})
    alphas.append(model.params['const'])
    betas.append(model.params['Mkt-Rf'])
    tstats_alpha.append(model.tvalues['const'])
    tstats_beta.append(model.tvalues['Mkt-Rf'])
    residuals.append(model.resid)

# Predicted and actual mean market return
mean_mkt_rf = data['Mkt-Rf'].mean()
predicted_means = np.array(betas) * mean_mkt_rf
actual_means = data[portfolio_excess_columns].mean().values

# Gibbons, Ross, Shanken test
N = len(portfolio_excess_columns) # num of portfolios
T = len(data) # num of obs

# Variance of the residuals
residuals_matrix = np.column_stack(residuals)
Sigma = np.cov(residuals_matrix, rowvar=False)

# Mean market excess return and variance
mean_mkt_rf = data['Mkt-Rf'].mean()
var_mkt_rf = data['Mkt-Rf'].var()

# GRS Statistic of alphas
# (note self): See page 5 https://ba-odegaard.no/teach/notes/gibbons_ross_shanken/grs_lecture.pdf
# or 14 in slides 
alpha_array = np.array(alphas)
Sigma_inv = np.linalg.inv(Sigma)
GRS_numerator = (T - N - 1) / N
GRS_denominator = 1 + (mean_mkt_rf ** 2) / var_mkt_rf
GRS_stat = GRS_numerator * (alpha_array.T @ Sigma_inv @ alpha_array) / GRS_denominator
GRS_p_value = 1 - stats.f.cdf(GRS_stat, N, T - N - 1)

print(f"GRS Statistic: {GRS_stat:.4f}")
print(f"GRS Test p-value: {GRS_p_value:.4f}")

# Newey-West and Wald of alphas
alpha_array = np.array(alphas)
# Collect the covariance matrices of alphas
cov_alphas = []
nw_se_alphas = []

for col in portfolio_excess_columns:
    y = data[col]
    x = data['Mkt-Rf']
    x = sm.add_constant(x)
    # Fit the OLS model with HAC (Newey-West) covariance
    model = sm.OLS(y, x).fit(cov_type='HAC', cov_kwds={'maxlags': 1})
    
    # Extract covariance for alpha
    cov_alpha = model.cov_params()['const']['const']
    cov_alphas.append(cov_alpha)
    
    # Calculate NW standard error for alpha
    nw_se_alpha = np.sqrt(cov_alpha)
    nw_se_alphas.append(nw_se_alpha)

# Convert covariances to a diagonal matrix
cov_alpha_matrix = np.diag(cov_alphas)

# Wald Statistic
wald_stat = alpha_array.T @ np.linalg.inv(cov_alpha_matrix) @ alpha_array
wald_p_value = 1 - stats.chi2.cdf(wald_stat, len(alpha_array))

# Save the Newey-West SEs
nw_se_dict = {col: se for col, se in zip(portfolio_excess_columns, nw_se_alphas)}

# Display results
print(f"\nWald Statistic: {wald_stat:.4f}")
print(f"Wald Test p-value: {wald_p_value:.4f}")
print(f"\nNewey-West Standard Errors for Alphas:\n{nw_se_dict}")

# Plot actual vs predicted mean excess returns
plt.figure(figsize=(8,6))
plt.scatter(predicted_means, actual_means)
plt.xlabel('Predicted Mean Excess Returns')
plt.ylabel('Actual Mean Excess Returns')
plt.title('Actual vs. Predicted Mean Excess Returns (CAPM)')
plt.plot([min(predicted_means), max(predicted_means)], [min(predicted_means), max(predicted_means)], 'r--')
plt.savefig('A5_1b.png', dpi=300)
plt.show()

# Calculate R-squared between actual and predicted means
correlation_matrix = np.corrcoef(predicted_means, actual_means)
r_squared = correlation_matrix[0,1]**2
print(f"R-squared between actual and predicted mean excess returns: {r_squared:.4f}")

# Alphas and t-statistics
alpha_results = pd.DataFrame({
    'Portfolio': portfolio_excess_columns,
    'Alpha': alphas,
    'Beta': betas,
    't-stat Alpha': tstats_alpha,
    't-stat Beta': tstats_beta
})

print("\nCAPM Regression Results:")
print(alpha_results[['Portfolio', 'Alpha', 't-stat Alpha']])

In [ ]:
# 1c)
# Using the Fama-French 3-factor model
selected_factors = ['Mkt-Rf', 'SMB', 'HML']

# Regression
alphas_mf = []
betas_mf = []
t_stat_betas = []
tstats_alpha_mf = []
residuals_mf = []
cov_alphas = []  # For storing alpha variances
nw_se_alphas = []  # For storing NW standard errors of alphas

for col in portfolio_excess_columns:
    y = data[col]
    X = data[selected_factors]
    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 1})
    
    # Store regression results
    alphas_mf.append(model.params['const'])
    betas = model.params[selected_factors].values
    betas_mf.append(betas)
    t_stat_betas.append(model.tvalues[selected_factors])
    tstats_alpha_mf.append(model.tvalues['const'])
    residuals_mf.append(model.resid)
    
    # Newey-West standard errors for alpha
    cov_alpha = model.cov_params().loc['const', 'const']
    cov_alphas.append(cov_alpha)
    nw_se_alpha = np.sqrt(cov_alpha)
    nw_se_alphas.append(nw_se_alpha)

# Calculate predicted mean
mean_factors = data[selected_factors].mean().values
betas_mf_array = np.array(betas_mf)  
predicted_means_mf = betas_mf_array @ mean_factors  

# GRS test
N = len(portfolio_excess_columns)
T = len(data)
K = len(selected_factors)  # Number of factors

# Variance of the residuals
residuals_matrix_mf = np.column_stack(residuals_mf)
Sigma_mf = np.cov(residuals_matrix_mf, rowvar=False)

# Mean factor returns and covariance matrix
mean_factors = data[selected_factors].mean().values
cov_factors = data[selected_factors].cov().values

# GRS Statistic for 3FF
alpha_array_mf = np.array(alphas_mf)
Sigma_inv_mf = np.linalg.inv(Sigma_mf)
GRS_numerator_mf = (T - N - K) / N
GRS_denominator_mf = 1 + mean_factors.T @ np.linalg.inv(cov_factors) @ mean_factors
GRS_stat_mf = GRS_numerator_mf * (alpha_array_mf.T @ Sigma_inv_mf @ alpha_array_mf) / GRS_denominator_mf
GRS_p_value_mf = 1 - stats.f.cdf(GRS_stat_mf, N, T - N - K)

print(f"\nGRS Statistic (Multi-Factor): {GRS_stat_mf:.4f}")
print(f"GRS Test p-value (Multi-Factor): {GRS_p_value_mf:.4f}")

# Wald test
# Stack alphas and compute covariance matrix
cov_alphas_mf = []
for col in portfolio_excess_columns:
    y = data[col]
    X = data[selected_factors]
    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags':1})
    cov_alpha = model.cov_params().loc['const', 'const']
    cov_alphas_mf.append(cov_alpha)
cov_alpha_matrix_mf = np.diag(cov_alphas_mf)

# Wald Statistic
wald_stat_mf = alpha_array_mf.T @ np.linalg.inv(cov_alpha_matrix_mf) @ alpha_array_mf
wald_p_value_mf = 1 - stats.chi2.cdf(wald_stat_mf, N)

print(f"\nWald Statistic (Multi-Factor): {wald_stat_mf:.4f}")
print(f"Wald Test p-value (Multi-Factor): {wald_p_value_mf:.4f}")

# PLot
plt.figure(figsize=(8,6))
plt.scatter(predicted_means_mf, actual_means)
plt.xlabel('Predicted Mean Excess Returns')
plt.ylabel('Actual Mean Excess Returns')
plt.title('Actual vs. Predicted Mean Excess Returns (Multi-Factor Model)')
plt.plot([min(predicted_means_mf), max(predicted_means_mf)], [min(predicted_means_mf), max(predicted_means_mf)], 'r--')
plt.savefig('A5_1c.png', dpi=300)
plt.show()

# R-squared between actual and predicted means
correlation_matrix_mf = np.corrcoef(predicted_means_mf, actual_means)
r_squared_mf = correlation_matrix_mf[0,1]**2
print(f"R-squared between actual and predicted mean excess returns (Multi-Factor): {r_squared_mf:.4f}")

# Convert betas_mf to DataFrame
betas_df = pd.DataFrame(betas_mf, columns=[f'Beta_{factor}' for factor in selected_factors])

# Create results DataFrame
alpha_results_mf = pd.DataFrame({
    'Portfolio': portfolio_excess_columns,
    'Alpha': alphas_mf,
    't-stat Alpha': tstats_alpha_mf,
    'NW SE Alpha': nw_se_alphas
}).join(betas_df) 

# Add t-statistics for betas
t_stats_df = pd.DataFrame(t_stat_betas, columns=[f't-stat {factor}' for factor in selected_factors])
alpha_results_mf = alpha_results_mf.join(t_stats_df)

# Display results
print("\nMulti-Factor Regression Results:")
print(alpha_results_mf)

In [ ]:
# 2a)
#
# Store betas
betas_capm = {}
betas_ff3 = {}

# First-Pass regression for CAPM
for col in portfolio_excess_columns:
    y = data[col]
    X = data['Mkt-Rf']
    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit()
    betas_capm[col] = model.params['Mkt-Rf']

# First-Pass regression for 3-factor Fama French
for col in portfolio_excess_columns:
    y = data[col]
    X = data[['Mkt-Rf', 'SMB', 'HML']]
    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit()
    betas_ff3[col] = model.params[['Mkt-Rf', 'SMB', 'HML']]

# Convert betas to DataFrame
betas_capm_df = pd.DataFrame.from_dict(betas_capm, orient='index', columns=['Beta_Mkt'])
betas_ff3_df = pd.DataFrame.from_dict(betas_ff3, orient='index')

######################
# Second step for CAPM
# Store lambdas
lambda_t_capm = []

# Time periods
time_periods = data.index

for t in time_periods:
    # Cross-sectional regression at time t
    # Dependent variable: Portfolio excess returns at time t
    y_t = data.loc[t, portfolio_excess_columns].values  # Shape (25,)
    
    # Independent variable: Betas from first-pass regression
    X_t = betas_capm_df['Beta_Mkt'].values.reshape(-1, 1)  # Shape (25,1)
    
    # Run cross-sectional regression
    model_t = sm.OLS(y_t, X_t).fit()
    
    # Store estimated lambda_t (market risk premium at time t)
    lambda_t_capm.append(model_t.params[0])

# Convert lambda_t to numpy array
lambda_t_capm = np.array(lambda_t_capm)

# Compute mean and stdev for CAPM
lambda_capm_mean = lambda_t_capm.mean()
lambda_capm_std = lambda_t_capm.std(ddof=1) / np.sqrt(len(lambda_t_capm))  # Standard error

print("Fama-MacBeth Estimation Results (CAPM):")
print(f"Estimated Market Risk Premium (lambda_M): {lambda_capm_mean:.6f}")
print(f"Standard Error: {lambda_capm_std:.6f}")
print(f"t-Statistic: {lambda_capm_mean / lambda_capm_std:.4f}")

# Predicted mean excess return
mu_hat = data[portfolio_excess_columns].mean().values  # Actual mean excess returns
betas_capm_array = betas_capm_df['Beta_Mkt'].values
mu_pred_capm = betas_capm_array * lambda_capm_mean  # Predicted mean excess returns

# Plot
plt.figure(figsize=(8,6))
plt.scatter(mu_pred_capm, mu_hat)
plt.xlabel('Predicted Mean Excess Returns')
plt.ylabel('Actual Mean Excess Returns')
plt.title('Actual vs. Predicted Mean Excess Returns (Fama-MacBeth CAPM)')
plt.plot([min(mu_pred_capm), max(mu_pred_capm)], [min(mu_pred_capm), max(mu_pred_capm)], 'r--')
plt.savefig('A5_2a_CAPM.png', dpi = 300)
plt.show()

# R-Squared
correlation_matrix_capm = np.corrcoef(mu_pred_capm, mu_hat)
r_squared_capm = correlation_matrix_capm[0,1]**2
print(f"R-squared (Fama-MacBeth CAPM): {r_squared_capm:.4f}")

##################
#### Same steps as above but for 3-factor model
lambda_t_ff3 = []

for t in time_periods:
    # Dependent variable: Portfolio excess returns at time t
    y_t = data.loc[t, portfolio_excess_columns].values  # Shape (25,)
    
    # Independent variables: Betas from first-pass regression
    X_t = betas_ff3_df.values  # Shape (25, 3)
    
    # Run cross-sectional regression
    model_t = sm.OLS(y_t, X_t).fit()
    
    # Store estimated lambda_t (risk premia at time t)
    lambda_t_ff3.append(model_t.params)

# Convert lambda_t_ff3 to DataFrame
lambda_t_ff3_df = pd.DataFrame(lambda_t_ff3, columns=['lambda_Mkt', 'lambda_SMB', 'lambda_HML'])

# Compute mean and stdev for FF3
lambda_ff3_mean = lambda_t_ff3_df.mean()
lambda_ff3_std = lambda_t_ff3_df.std(ddof=1) / np.sqrt(len(lambda_t_ff3_df))  # Standard errors

print("\nFama-MacBeth Estimation Results (Fama-French 3-Factor Model):")
print(lambda_ff3_mean)
print("\nStandard Errors:")
print(lambda_ff3_std)
print("\nt-Statistics:")
print(lambda_ff3_mean / lambda_ff3_std)

# Predicted mean excess return
betas_ff3_array = betas_ff3_df.values  # Shape (25, 3)
lambda_ff3_mean_array = lambda_ff3_mean.values  # Shape (3,)
mu_pred_ff3 = betas_ff3_array @ lambda_ff3_mean_array  # Matrix multiplication

# Plot
plt.figure(figsize=(8,6))
plt.scatter(mu_pred_ff3, mu_hat)
plt.xlabel('Predicted Mean Excess Returns')
plt.ylabel('Actual Mean Excess Returns')
plt.title('Actual vs. Predicted Mean Excess Returns (Fama-MacBeth FF3)')
plt.plot([min(mu_pred_ff3), max(mu_pred_ff3)], [min(mu_pred_ff3), max(mu_pred_ff3)], 'r--')
plt.savefig('A5_2a_FF3.png', dpi = 300)
plt.show()

# R-squared
correlation_matrix_ff3 = np.corrcoef(mu_pred_ff3, mu_hat)
r_squared_ff3 = correlation_matrix_ff3[0,1]**2
print(f"R-squared (Fama-MacBeth FF3): {r_squared_ff3:.4f}")

In [ ]:
# 2b) Same as 2a) but add constant in the second pastt

# CAPM
# Store lambdas
lambda_0_t_capm = []
lambda_t_capm_intercept = []

for t in time_periods:
    y_t = data.loc[t, portfolio_excess_columns].values
    X_t = pd.DataFrame({
        'const': 1,  # Add intercept explicitly
        'Beta_Mkt': betas_capm_df['Beta_Mkt']
    })
    
    model_t = sm.OLS(y_t, X_t).fit()
    
    # Save lambdas
    lambda_0_t_capm.append(model_t.params['const'])
    lambda_t_capm_intercept.append(model_t.params['Beta_Mkt'])

# Convert to numpy arrays
lambda_0_t_capm = np.array(lambda_0_t_capm)
lambda_t_capm_intercept = np.array(lambda_t_capm_intercept)

# Compute Average Risk Premia and Standard Errors
lambda_0_capm_mean = lambda_0_t_capm.mean()
lambda_0_capm_std = lambda_0_t_capm.std(ddof=1) / np.sqrt(len(lambda_0_t_capm))
t_stat_lambda_0 = lambda_0_capm_mean / lambda_0_capm_std

lambda_capm_mean_intercept = lambda_t_capm_intercept.mean()
lambda_capm_std_intercept = lambda_t_capm_intercept.std(ddof=1) / np.sqrt(len(lambda_t_capm_intercept))
t_stat_lambda = lambda_capm_mean_intercept / lambda_capm_std_intercept

print("\nFama-MacBeth Estimation Results with Intercept (CAPM):")
print(f"Estimated Intercept (lambda_0): {lambda_0_capm_mean:.6f}")
print(f"Standard Error: {lambda_0_capm_std:.6f}")
print(f"t-Statistic: {t_stat_lambda_0:.4f}")

print(f"\nEstimated Market Risk Premium (lambda_M): {lambda_capm_mean_intercept:.6f}")
print(f"Standard Error: {lambda_capm_std_intercept:.6f}")
print(f"t-Statistic: {t_stat_lambda:.4f}")

# Test the hypothesis that lambda_0 = 0
p_value_lambda_0 = 2 * (1 - stats.t.cdf(np.abs(t_stat_lambda_0), df=len(lambda_0_t_capm)-1))
print(f"\np-value for test of lambda_0 = 0: {p_value_lambda_0:.4f}")

# Compute Predicted Mean Excess Returns with Intercept
mu_pred_capm_intercept = lambda_0_capm_mean + betas_capm_array * lambda_capm_mean_intercept

# Plot Actual vs. Predicted Mean Excess Returns
plt.figure(figsize=(8,6))
plt.scatter(mu_pred_capm_intercept, mu_hat)
plt.xlabel('Predicted Mean Excess Returns')
plt.ylabel('Actual Mean Excess Returns')
plt.title('Actual vs. Predicted Mean Excess Returns (Fama-MacBeth CAPM with Intercept)')
plt.plot([min(mu_pred_capm_intercept), max(mu_pred_capm_intercept)], [min(mu_pred_capm_intercept), max(mu_pred_capm_intercept)], 'r--')
plt.savefig('A5_2b_capm.png', dpi = 300)
plt.show()

# R-squared
correlation_matrix_capm_intercept = np.corrcoef(mu_pred_capm_intercept, mu_hat)
r_squared_capm_intercept = correlation_matrix_capm_intercept[0,1]**2
print(f"R-squared (Fama-MacBeth CAPM with Intercept): {r_squared_capm_intercept:.4f}")

## 3FF ##
lambda_0_t_ff3 = []
lambda_t_ff3_intercept = []

for t in time_periods:
    y_t = data.loc[t, portfolio_excess_columns].values 

    X_t = pd.DataFrame(betas_ff3_df)
    X_t.insert(0, 'const', 1)  # Add intercept as the first column
    X_t = X_t.loc[data.loc[t, portfolio_excess_columns].index]
    
    # Check for NaN values
    if X_t.isnull().values.any() or np.isnan(y_t).any():
        print(f"Skipping time period {t} due to missing data")
        continue

    model_t = sm.OLS(y_t, X_t).fit()
    
    # Save lambdas
    lambda_0_t_ff3.append(model_t.params['const'])
    lambda_t_ff3_intercept.append(model_t.params.drop('const'))  # Exclude intercept

# Convert to DataFrame without specifying columns
lambda_t_ff3_intercept = pd.DataFrame(lambda_t_ff3_intercept)

# Rename the columns
lambda_t_ff3_intercept.rename(columns={'Mkt-Rf': 'lambda_Mkt', 'SMB': 'lambda_SMB', 'HML': 'lambda_HML'}, inplace=True)

# Now proceed to compute statistics
lambda_0_t_ff3 = np.array(lambda_0_t_ff3)

# Compute Average Risk Premia and Standard Errors
lambda_0_ff3_mean = lambda_0_t_ff3.mean()
lambda_0_ff3_std = lambda_0_t_ff3.std(ddof=1) / np.sqrt(len(lambda_0_t_ff3))
t_stat_lambda_0_ff3 = lambda_0_ff3_mean / lambda_0_ff3_std

lambda_ff3_mean_intercept = lambda_t_ff3_intercept.mean()
lambda_ff3_std_intercept = lambda_t_ff3_intercept.std(ddof=1) / np.sqrt(len(lambda_t_ff3_intercept))
t_stat_lambda_ff3 = lambda_ff3_mean_intercept / lambda_ff3_std_intercept

print("\nFama-MacBeth Estimation Results with Intercept (Fama-French 3-Factor Model):")
print(f"Estimated Intercept (lambda_0): {lambda_0_ff3_mean:.6f}")
print(f"Standard Error: {lambda_0_ff3_std:.6f}")
print(f"t-Statistic: {t_stat_lambda_0_ff3:.4f}")

print("\nEstimated Risk Premia:")
print(lambda_ff3_mean_intercept)
print("\nStandard Errors:")
print(lambda_ff3_std_intercept)
print("\nt-Statistics:")
print(t_stat_lambda_ff3)

# Test the hypothesis that lambda_0 = 0
p_value_lambda_0_ff3 = 2 * (1 - stats.t.cdf(np.abs(t_stat_lambda_0_ff3), df=len(lambda_0_t_ff3)-1))
print(f"\np-value for test of lambda_0 = 0 (FF3): {p_value_lambda_0_ff3:.4f}")

# Predicted Mean Excess Returns with Intercept
mu_pred_ff3_intercept = lambda_0_ff3_mean + betas_ff3_array @ lambda_ff3_mean_intercept.values

# Plot Actual vs. Predicted Mean Excess Returns (FF3 with Intercept)
plt.figure(figsize=(8,6))
plt.scatter(mu_pred_ff3_intercept, mu_hat)
plt.xlabel('Predicted Mean Excess Returns')
plt.ylabel('Actual Mean Excess Returns')
plt.title('Actual vs. Predicted Mean Excess Returns (Fama-MacBeth FF3 with Intercept)')
plt.plot([mu_pred_ff3_intercept.min(), mu_pred_ff3_intercept.max()], 
         [mu_pred_ff3_intercept.min(), mu_pred_ff3_intercept.max()], 'r--')
plt.savefig('A5_2b_3ff.png', dpi = 300)
plt.show()

# R-squared
correlation_matrix_ff3_intercept = np.corrcoef(mu_pred_ff3_intercept, mu_hat)
r_squared_ff3_intercept = correlation_matrix_ff3_intercept[0,1]**2
print(f"R-squared (Fama-MacBeth FF3 with Intercept): {r_squared_ff3_intercept:.4f}")

In [ ]:
# 2c)

# Number of portfolios
N = betas_capm_df.shape[0]

## CAPM
# Compute the covariance matrix of betas (from first-pass)
cov_betas_capm = np.diag(data[portfolio_excess_columns].apply(lambda x: sm.OLS(x, sm.add_constant(data['Mkt-Rf'])).fit().bse['Mkt-Rf'], axis=0)**2)

# Variance of lambda estimates from second pass
T = len(lambda_t_capm)
var_lambda_capm = np.var(lambda_t_capm, ddof=1) / T

# Shanken Correction: Adjust the variance of lambda_hat
# Var(lambda_hat) = var_lambda_capm + (beta_hat' beta_hat)^{-1} * beta_hat' Cov(beta_hat) beta_hat * (beta_hat' beta_hat)^{-1} * lambda_hat^2
beta_hat_capm = betas_capm_df['Beta_Mkt'].values
beta_hat_matrix_capm = np.vstack(beta_hat_capm).T  
beta_beta_capm = beta_hat_matrix_capm.T @ beta_hat_matrix_capm 

# Since cov_betas_capm is diagonal, beta_hat' Cov(beta_hat) beta_hat = sum(beta_i^2 * Var(beta_i))
beta_cov_beta_capm = (beta_hat_capm**2 * np.diag(cov_betas_capm)).sum()

# Compute the correction term
correction_term_capm = (1 / (beta_beta_capm[0,0]**2)) * beta_cov_beta_capm

# Total variance with Shanken correction
var_lambda_capm_corrected = var_lambda_capm + correction_term_capm

# Corrected standard error
lambda_capm_std_corrected = np.sqrt(var_lambda_capm_corrected)

print("Shanken-Corrected Standard Error (CAPM):", lambda_capm_std_corrected)
print("Shanken-Corrected t-Statistic (CAPM):", lambda_capm_mean / lambda_capm_std_corrected)

## FF3
# Number of factors in FF3
K_ff3 = betas_ff3_df.shape[1]

# Compute the covariance matrix of betas (from first-pass)
cov_betas_ff3 = np.diag(data[portfolio_excess_columns].apply(lambda x: sm.OLS(x, sm.add_constant(data[['Mkt-Rf', 'SMB', 'HML']])).fit().bse[['Mkt-Rf', 'SMB', 'HML']].values, axis=0).sum())

# Compute the covariance matrix of lambda estimates across factors
var_lambda_ff3 = lambda_t_ff3_df.var() / T  # Shape (3,)

# Extract betas and multiply
beta_hat_ff3 = betas_ff3_df.values  
beta_beta_ff3 = beta_hat_ff3.T @ beta_hat_ff3

# Compute covariance of betas
# Here, we approximate Cov(beta_hat_ff3) as cov_betas_ff3 for each factor
cov_beta_hat_ff3 = np.linalg.inv(beta_beta_ff3) @ (beta_hat_ff3.T @ cov_betas_ff3 @ beta_hat_ff3) @ np.linalg.inv(beta_beta_ff3)

# Shanken Correction: Adjust the variance of lambda_hat
# Var(lambda_hat) = Var(lambda_hat_ff3) + Trace(beta_beta_ff3^{-1} Cov(beta_hat_ff3))
var_lambda_ff3_corrected = var_lambda_ff3 + np.diag(np.linalg.inv(beta_beta_ff3) @ cov_beta_hat_ff3)

# Corrected standard errors
lambda_ff3_std_corrected = np.sqrt(var_lambda_ff3_corrected)

print("\nShanken-Corrected Standard Errors (FF3):")
print(lambda_ff3_std_corrected)
print("\nShanken-Corrected t-Statistics (FF3):")
print(lambda_ff3_mean / lambda_ff3_std_corrected)


In [ ]:
# 2d)
# For 2d, we apply the GRS test to see if alphas are equal to zero in our CAPM and 3factor model


# First-Pass regression for CAPM
alphas_capm = []
residuals_capm = []

for col in portfolio_excess_columns:
    y = data[col]
    X = data['Mkt-Rf']
    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit()
    alphas_capm.append(model.params['const'])
    residuals_capm.append(model.resid)

# Convert alphas and residuals to arrays
alphas_capm = np.array(alphas_capm)
residuals_capm = np.array(residuals_capm)

# Residual covariance matrix
Sigma_capm = np.cov(residuals_capm)

# Average squared residuals
sigma2_e_capm = np.mean(np.diag(Sigma_capm))

# Mean of the factors
mean_factors_capm = data['Mkt-Rf'].mean()

# Calculate GRS Statistic
numerator_capm = (T - N - K) / N
inverse_Sigma_capm = np.linalg.inv(Sigma_capm)
alpha_term_capm = alphas_capm @ inverse_Sigma_capm @ alphas_capm
denominator_capm = 1 + (mean_factors_capm ** 2) / (K * sigma2_e_capm)

F_grs_capm = numerator_capm * alpha_term_capm / denominator_capm

# Degrees of freedom
df1_capm = N
df2_capm = T - N - K

# Calculate p-value
p_value_capm = 1 - stats.f.cdf(F_grs_capm, df1_capm, df2_capm)

print("\nGRS Test Results for CAPM:")
print(f"GRS Statistic: {F_grs_capm:.4f}")
print(f"Degrees of Freedom: ({df1_capm}, {df2_capm})")
print(f"p-value: {p_value_capm:.4f}")

#########################
# GRS Test for FF3 Model
#########################

# First-Pass regression for FF3
alphas_ff3 = []
residuals_ff3 = []

for col in portfolio_excess_columns:
    y = data[col]
    X = data[['Mkt-Rf', 'SMB', 'HML']]
    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit()
    alphas_ff3.append(model.params['const'])
    residuals_ff3.append(model.resid)

# Convert alphas and residuals to arrays
alphas_ff3 = np.array(alphas_ff3)
residuals_ff3 = np.array(residuals_ff3)

# Residual covariance matrix
Sigma_ff3 = np.cov(residuals_ff3)

# Average squared residuals
sigma2_e_ff3 = np.mean(np.diag(Sigma_ff3))

# Mean of the factors
mean_factors_ff3 = data[['Mkt-Rf', 'SMB', 'HML']].mean().values

#Calculate terms
factor_cov_ff3 = np.cov(data[['Mkt-Rf', 'SMB', 'HML']].T)
term_ff3 = mean_factors_ff3 @ np.linalg.inv(factor_cov_ff3) @ mean_factors_ff3

# Calculate GRS Statistic
numerator_ff3 = (T - N - K) / N
inverse_Sigma_ff3 = np.linalg.inv(Sigma_ff3)
alpha_term_ff3 = alphas_ff3 @ inverse_Sigma_ff3 @ alphas_ff3
denominator_ff3 = 1 + (term_ff3) / sigma2_e_ff3

F_grs_ff3 = numerator_ff3 * alpha_term_ff3 / denominator_ff3

# Degrees of freedom
df1_ff3 = N
df2_ff3 = T - N - K

# Compute p-value
p_value_ff3 = 1 - stats.f.cdf(F_grs_ff3, df1_ff3, df2_ff3)

print("\nGRS Test Results for Fama-French 3-Factor Model:")
print(f"GRS Statistic: {F_grs_ff3:.4f}")
print(f"Degrees of Freedom: ({df1_ff3}, {df2_ff3})")
print(f"p-value: {p_value_ff3:.4f}")